# ⚡ AETHER All-in-One Studio — Google Colab & Kaggle Runner

**Models:** Stable Diffusion XL (Images) + Fish Audio S2 Pro (Voice)

This notebook runs **BOTH** models simultaneously on a single free T4 GPU using a unified API tunnel and CPU offloading!

> ⚠️ **Enable GPU before running!**  
> `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1. Install Dependencies & Clone Models
# --- Environment + storage detection (Colab vs Kaggle vs local) ---
# Kaggle has no /content, and caps /kaggle/working at 20 GB. The S2 Pro model
# does not fit there via git-lfs (which stores every file twice), so the model
# goes to whichever mount actually has free space and is symlinked in.
import os, shutil

if os.path.isdir('/kaggle'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
FISH_DIR = os.path.join(BASE, 'fish-speech')
os.makedirs(BASE, exist_ok=True)

# Pick the roomiest scratch mount for the ~10 GB of model weights.
_cands = []
for _c in ('/kaggle/temp', '/tmp', os.path.join(BASE, '.scratch')):
    try:
        os.makedirs(_c, exist_ok=True)
        _cands.append((shutil.disk_usage(_c).free, _c))
    except Exception:
        pass
SCRATCH = max(_cands)[1] if _cands else BASE
os.environ['HF_HOME'] = os.path.join(SCRATCH, 'hf')

def disk_report(label=''):
    print(f'💾 Disk {label}')
    for _p in dict.fromkeys([BASE, SCRATCH, '/']):
        try:
            _t, _u, _f = shutil.disk_usage(_p)
            print(f'   {_p:<24} {_f/1e9:6.1f} GB free / {_t/1e9:6.1f} GB total')
        except Exception:
            pass

print(f'📁 Environment base: {BASE}')
print(f'📁 Fish Speech dir : {FISH_DIR}')
print(f'📁 Model scratch   : {SCRATCH}')

disk_report('before install')

os.chdir(BASE)

# Remove any half-finished previous attempt so a re-run starts clean and frees
# the space that a failed run left behind.
!rm -rf "{FISH_DIR}"
!pip cache purge 2>/dev/null || true

!apt-get update -qq
!apt-get install -y portaudio19-dev build-essential rustc cargo git git-lfs psmisc

!git clone https://github.com/fishaudio/fish-speech.git "{FISH_DIR}"
os.chdir(FISH_DIR)
print(f'✅ Working directory is now: {os.getcwd()}')

# --- APPLY MEMORY OOM FIX ---
# Force bfloat16 at init: PyTorch's float32 default needs ~20 GB and crashes a
# free T4/P100. bfloat16 halves it.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
llama_path = os.path.join(FISH_DIR, "fish_speech/models/text2semantic/llama.py")
with open(llama_path, "r") as f:
    code = f.read()

def must_replace(text, old, new, what):
    """Fail loudly if an upstream anchor moved.

    These patches previously used str.replace and printed success regardless,
    so when fish-speech changed its source the notebook happily reported a fix
    it had not made.
    """
    if old not in text:
        raise RuntimeError(
            f"Cannot apply patch '{what}': anchor not found in the current "
            "fish-speech source. Upstream has changed; the patch needs updating."
        )
    return text.replace(old, new)

code = must_replace(
    code,
    "model = model_cls(config)",
    "torch.set_default_dtype(torch.bfloat16)\n        model = model_cls(config)\n        torch.set_default_dtype(torch.float32)",
    "bfloat16 init",
)

# Restrict max sequence length (the KV cache is preallocated for this many
# tokens, so it is also the memory dial).
#
# DO NOT set this to 2048 or lower. generate_long() in
# fish_speech/models/text2semantic/inference.py rejects a prompt when
#     encoded.size(1) > max_length - 2048
# so it reserves 2048 tokens for generation. At max_seq_len=2048 the prompt
# budget is exactly zero and the server dies during its own warm-up with
# "Prompt is too long: 23 > 0", before serving a single request.
#
# 3072 leaves 1024 tokens for the prompt; observed reference prompts run
# ~390. If CUDA OOM returns on long chunks, shorten the reference clips
# (14s -> ~8s cuts the prompt from ~302 to ~187 tokens) rather than lowering
# this value.
code = must_replace(
    code,
    "config = BaseModelArgs.from_pretrained(str(path))",
    "config = BaseModelArgs.from_pretrained(str(path))\n        config.max_seq_len = 3072",
    "max_seq_len cap",
)
with open(llama_path, "w") as f:
    f.write(code)
print("✅ Verified: bfloat16 init + max_seq_len cap applied to Fish Speech source.")
# ----------------------------

# Upgrade pip to ensure it pulls binary wheels instead of building from source
!python -m pip install -q --upgrade pip wheel
!pip install -q tokenizers transformers "huggingface_hub>=0.23"

# Fix protobuf/tensorflow crash on Kaggle by removing tensorflow (we only use PyTorch!)
!pip uninstall -y tensorflow
!pip install -q -U protobuf

# Install safely and FORCE torchvision downgrade to match Fish Speech's PyTorch version
!pip install -q -e . torchvision
!pip install -q accelerate torch fastapi uvicorn httpx pyngrok nest_asyncio pyrootutils psutil

# --- DOWNLOAD THE S2 PRO MODEL (single copy) ---
# NOT `git clone` — git-lfs keeps the blob in .git/lfs/objects *and* in the
# working tree, doubling ~10 GB of weights and exhausting Kaggle's 20 GB quota.
from huggingface_hub import snapshot_download

# cache_dir is passed explicitly: huggingface_hub resolves HF_HOME at import
# time, so the env var alone is unreliable if transformers was already imported.
HF_CACHE = os.path.join(SCRATCH, "hf")
print(f"⬇️  Downloading fishaudio/s2-pro into {HF_CACHE} (single copy)...")
model_path = snapshot_download(repo_id="fishaudio/s2-pro", cache_dir=HF_CACHE)
print(f"✅ Model at: {model_path}")

ckpt_link = os.path.join(FISH_DIR, "checkpoints", "s2-pro")
os.makedirs(os.path.dirname(ckpt_link), exist_ok=True)
if os.path.islink(ckpt_link):
    os.unlink(ckpt_link)
elif os.path.isdir(ckpt_link):
    shutil.rmtree(ckpt_link)
os.symlink(model_path, ckpt_link)
print(f"🔗 Linked {ckpt_link} -> {model_path}")

_codec = os.path.join(ckpt_link, "codec.pth")
print(f"🔍 codec.pth present: {os.path.exists(_codec)}")
if not os.path.exists(_codec):
    print("⚠️  codec.pth missing — the API server will fail to start.")
# ----------------------------

print("✅ Dependencies and Model installed!")

# NOTE: the old "VRAM leak fix" that rewrote tools/server/views.py has been
# removed. It searched for StreamingResponse(generator(), ...), which no longer
# exists upstream (it is StreamResponse now), so it silently did nothing while
# printing a success message. Current fish-speech already calls
# torch.cuda.empty_cache() itself in fish_speech/inference_engine/__init__.py.

disk_report('after install')

In [ ]:
# 2. Authenticate ngrok
# REPLACE "YOUR_TOKEN_HERE" WITH YOUR ACTUAL NGROK TOKEN IF SECRETS ARE NOT WORKING
MANUAL_TOKEN = ""

try:
    from google.colab import userdata
    colab_env = True
except ImportError:
    colab_env = False

try:
    if MANUAL_TOKEN:
        ngrok_token = MANUAL_TOKEN
    elif colab_env:
        ngrok_token = userdata.get("NGROK_TOKEN")
    else:
        # For Kaggle or other envs without Colab userdata
        import os
        ngrok_token = os.environ.get("NGROK_TOKEN", "")
        
    if not ngrok_token:
        raise ValueError("Token is empty!")
        
    !ngrok authtoken {ngrok_token}
    print("✅ ngrok authenticated")
except Exception as e:
    print("\n❌ FATAL ERROR: Could not authenticate with ngrok!")
    print("You have two options to fix this:")
    print("1. Paste your token between the quotes in MANUAL_TOKEN = \"\" at the top of this cell.")
    print("2. OR Add your ngrok token to Colab Secrets (the 🔑 icon on the left) as NGROK_TOKEN and turn the toggle switch ON.")
    raise Exception("STOPPING: You must provide a valid ngrok token before continuing!")

# 3. Add Custom Voices (Zero-Shot Cloning)
Fish Speech S2 Pro allows you to instantly clone ANY voice just by providing a 10-second reference audio file!

**How to clone your own voice:**
1. In the file explorer on the left, navigate to the `references/` folder inside your `fish-speech` directory (`/content/fish-speech/references/` on Colab, `/kaggle/working/fish-speech/references/` on Kaggle — the install cell prints the exact path)
2. Create a new folder with your voice name (e.g. `My_Voice`)
3. Upload a 10-20s clean `.wav` or `.mp3` file of the voice speaking and name it `audio.wav`
4. Create a text file called `audio.lab` in the same folder and type out exactly what is being said in the audio.
5. Restart the Unified API Server cell below so it detects the new folder!

*(Run the cell below to install the curated **AETHERSTUDIO Premium Voice Pack** — 8 studio-grade references. Every clip was measured for sample rate, spectral rolloff and noise floor before inclusion, and each transcript is generated by Whisper from the trimmed audio so the `.lab` always matches the `.wav`.)*

In [ ]:
# 3. Install the AETHERSTUDIO Premium Voice Pack
# Every clip below was measured before inclusion (sample rate, 95% spectral
# rolloff, high-frequency energy, noise floor). The previous pack failed for two
# reasons, both fixed here:
#
#   1. Band-limited sources. jfk.wav rolled off at 3.3 kHz and ted_60_16k.wav had
#      0.01% energy above 8 kHz -- Fish clones that muffled character faithfully.
#   2. Fabricated transcripts. Three voices shared the placeholder line
#      "I am a high quality human reference voice used for cloning.", which is not
#      what those recordings say. Fish uses the reference text for in-context
#      learning, so a wrong .lab degrades output badly.
#
# Transcripts are now produced by Whisper from the actual (trimmed) audio, so the
# .lab can never drift from the .wav again.
# --- Environment + storage detection (Colab vs Kaggle vs local) ---
# Kaggle has no /content, and caps /kaggle/working at 20 GB. The S2 Pro model
# does not fit there via git-lfs (which stores every file twice), so the model
# goes to whichever mount actually has free space and is symlinked in.
import os, shutil

if os.path.isdir('/kaggle'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
FISH_DIR = os.path.join(BASE, 'fish-speech')
os.makedirs(BASE, exist_ok=True)

# Pick the roomiest scratch mount for the ~10 GB of model weights.
_cands = []
for _c in ('/kaggle/temp', '/tmp', os.path.join(BASE, '.scratch')):
    try:
        os.makedirs(_c, exist_ok=True)
        _cands.append((shutil.disk_usage(_c).free, _c))
    except Exception:
        pass
SCRATCH = max(_cands)[1] if _cands else BASE
os.environ['HF_HOME'] = os.path.join(SCRATCH, 'hf')

def disk_report(label=''):
    print(f'💾 Disk {label}')
    for _p in dict.fromkeys([BASE, SCRATCH, '/']):
        try:
            _t, _u, _f = shutil.disk_usage(_p)
            print(f'   {_p:<24} {_f/1e9:6.1f} GB free / {_t/1e9:6.1f} GB total')
        except Exception:
            pass

print(f'📁 Environment base: {BASE}')
print(f'📁 Fish Speech dir : {FISH_DIR}')
print(f'📁 Model scratch   : {SCRATCH}')

import os, shutil
import requests

if not os.path.isdir(FISH_DIR):
    raise RuntimeError(f'{FISH_DIR} does not exist - run the install cell (cell 1) first.')

_free_mb = shutil.disk_usage(FISH_DIR).free / 1e6
if _free_mb < 200:
    disk_report('current')
    raise RuntimeError(
        f'Only {_free_mb:.0f} MB free on {FISH_DIR} - not enough for the voice pack.'
    )

TARGET_SR   = 44100   # Fish S2 Pro is happiest at full-band 44.1 kHz
TARGET_SEC  = 14.0    # ~10-20 s is the sweet spot for reference cloning
MIN_SEC     = 4.0

HF = "https://huggingface.co"

# fallback_text is only used if Whisper is unavailable, and only where the true
# transcript is known with certainty. Voices without one are skipped rather than
# installed with a guessed .lab.
VOICES = {
    "Aria_Female_Warm": {
        "url": f"{HF}/spaces/coqui/xtts/resolve/main/examples/female.wav",
        "fallback_text": "This is a great day to learn something new about artificial intelligence.",
        "note": "48 kHz studio, SNR 63 dB - warm conversational female",
    },
    "Marcus_Male_Deep": {
        "url": f"{HF}/spaces/coqui/xtts/resolve/main/examples/male.wav",
        "fallback_text": "When I wake up, I expect a coffee ready and waiting for me.",
        "note": "48 kHz studio, SNR 58 dB - deep natural male",
    },
    "Sophia_Female_Narrator": {
        "url": "https://raw.githubusercontent.com/coqui-ai/TTS/dev/tests/data/ljspeech/wavs/LJ001-0001.wav",
        "fallback_text": (
            "Printing, in the only sense with which we are at present concerned, "
            "differs from most if not from all the arts and crafts represented in the Exhibition"
        ),
        "note": "LJSpeech audiobook narrator - measured clean, proven reference",
    },
    "Nova_Bright_Studio": {
        "url": f"{HF}/spaces/myshell-ai/OpenVoiceV2/resolve/main/examples/speaker0.mp3",
        "fallback_text": None,
        "note": "SNR 63.5 dB, rolloff 11.2 kHz - brightest measured; auto-trimmed",
    },
    "Ember_Expressive": {
        "url": f"{HF}/spaces/myshell-ai/OpenVoiceV2/resolve/main/examples/speaker2.mp3",
        "fallback_text": None,
        "note": "rolloff 12.4 kHz, 14.9% HF energy - most open/airy tone",
    },
    "Atlas_Documentary": {
        "url": f"{HF}/spaces/mrfakename/E2-F5-TTS/resolve/main/samples/main.flac",
        "fallback_text": None,
        "note": "rolloff 11.5 kHz - crisp documentary delivery",
    },
    "River_Natural": {
        "url": f"{HF}/spaces/mrfakename/E2-F5-TTS/resolve/main/src/f5_tts/infer/examples/basic/basic_ref_en.wav",
        "fallback_text": None,
        "note": "SNR 37.7 dB, rolloff 8.8 kHz - relaxed natural read",
    },
    "Sage_Storyteller": {
        "url": f"{HF}/spaces/mrfakename/E2-F5-TTS/resolve/main/samples/country.flac",
        "fallback_text": None,
        "note": "rolloff 8.3 kHz - measured storyteller cadence",
    },
}

# Voices deliberately dropped from the old pack, with the measurement that failed
# them. Kept here so the reasoning is visible rather than silently lost.
REMOVED = {
    "JFK_Presidential":    "95% rolloff 3.3 kHz (telephone-band archival) + real public figure",
    "MLK_Historical":      "1960s archival recording + real public figure",
    "TED_Talk_Speaker":    "16 kHz source, 0.01% energy above 8 kHz, 60 s long",
    "OpenVoice_Speaker_0": "SNR 29.7 dB and the .lab transcript did not match the audio",
    "OpenVoice_Speaker_1": "byte-identical to OpenVoice_Speaker_0, same wrong transcript",
    "OpenVoice_Speaker_2": "95% rolloff 4.5 kHz (dull), transcript did not match",
    "French_Speaker":      "non-English + placeholder transcript",
    "XTTS_Male_1":         "superseded by Marcus_Male_Deep (same source, now auto-trimmed)",
    "XTTS_Female_1":       "superseded by Aria_Female_Warm (same source, now auto-trimmed)",
    "Professional_Female": "superseded by Sophia_Female_Narrator (same source)",
}

!pip install -q faster-whisper soundfile librosa 2>/dev/null

import numpy as np
import soundfile as sf
import librosa

# --- Whisper: the transcripts must come from the audio, never be hand-written ---
_whisper = None
try:
    from faster_whisper import WhisperModel
    # CPU on purpose. The T4 has 14.56 GiB and Fish needs ~13.8 GiB of it, so a
    # Whisper model left resident here is exactly what pushed the decoder into
    # "CUDA out of memory". Transcribing 8 short clips on CPU costs a minute or
    # two, once, and removes the contention entirely.
    _whisper = WhisperModel("small.en", device="cpu", compute_type="int8",
                            download_root=os.path.join(SCRATCH, "whisper"))
    print("✅ Whisper (small.en) loaded on CPU - transcripts will match the audio.")
    print("   (CPU is deliberate: the GPU is reserved for Fish Speech.)")
except Exception as e:
    print(f"⚠️  Whisper unavailable ({e}).")
    print("   Only voices with a known-correct transcript will be installed.")


def load_mono(path):
    y, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    return y


def best_window(y, sr, target_sec=TARGET_SEC):
    """Pick the most speech-dense contiguous window, snapped to quiet boundaries.

    A reference should be one continuous stretch of clean speech. Long clips
    (speaker0.mp3 is ~59 s) otherwise waste context and can span pauses or
    changes in delivery.
    """
    need = int(target_sec * sr)
    if len(y) <= need:
        return y
    hop = int(0.05 * sr)
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    win = max(1, need // hop)
    csum = np.concatenate([[0.0], np.cumsum(rms)])
    means = (csum[win:] - csum[:-win]) / win
    start_f = int(np.argmax(means))
    # snap the start back to the quietest frame just before it (a natural pause)
    lo = max(0, start_f - int(0.6 * sr / hop))
    if start_f > lo:
        start_f = lo + int(np.argmin(rms[lo:start_f + 1]))
    s = start_f * hop
    return y[s:s + need]


def normalize(y, peak_dbfs=-1.0):
    p = float(np.max(np.abs(y))) or 1.0
    return y * ((10 ** (peak_dbfs / 20.0)) / p)


def measure(y, sr):
    """Same metrics used to select these sources, reported per installed voice."""
    S = np.abs(librosa.stft(y, n_fft=2048))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
    spec = S.mean(axis=1)
    total = spec.sum() or 1.0
    _i = min(int(np.searchsorted(np.cumsum(spec), 0.95 * total)), len(freqs) - 1)
    roll = float(freqs[_i])
    hf = float(spec[freqs >= 8000].sum() / total * 100)
    frames = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0]
    frames = np.sort(frames)
    noise = frames[max(0, int(len(frames) * 0.1))] or 1e-9
    snr = float(20 * np.log10((frames[-1] or 1e-9) / noise))
    return roll, hf, snr


ref_root = os.path.join(FISH_DIR, "references")
# Remove the entire old pack so retired voices cannot linger in the API listing.
for old in REMOVED:
    shutil.rmtree(os.path.join(ref_root, old), ignore_errors=True)
os.makedirs(ref_root, exist_ok=True)

print("\n🗑  Retired from the old pack:")
for k, why in REMOVED.items():
    print(f"   - {k}: {why}")

print("\n🎙️  Installing AETHERSTUDIO Premium Voice Pack...\n")
installed, skipped = [], []

for vid, meta in VOICES.items():
    vdir = os.path.join(ref_root, vid)
    os.makedirs(vdir, exist_ok=True)
    tmp = os.path.join(vdir, "_src" + os.path.splitext(meta["url"])[1])
    try:
        r = requests.get(meta["url"], timeout=120)
        r.raise_for_status()
        with open(tmp, "wb") as f:
            f.write(r.content)

        y = load_mono(tmp)
        if len(y) / TARGET_SR < MIN_SEC:
            raise ValueError(f"only {len(y)/TARGET_SR:.1f}s of audio")
        y = normalize(best_window(y, TARGET_SR))

        wav_path = os.path.join(vdir, "audio.wav")
        sf.write(wav_path, y, TARGET_SR, subtype="PCM_16")

        text = None
        if _whisper is not None:
            segs, _ = _whisper.transcribe(wav_path, language="en", beam_size=5)
            text = " ".join(s.text.strip() for s in segs).strip()
        if not text:
            text = meta["fallback_text"]
        if not text:
            raise ValueError("no reliable transcript (Whisper unavailable, no known text)")

        with open(os.path.join(vdir, "audio.lab"), "w", encoding="utf-8") as f:
            f.write(text)

        roll, hf, snr = measure(y, TARGET_SR)
        installed.append(vid)
        print(f"✅ {vid}")
        print(f"     {meta['note']}")
        print(f"     {len(y)/TARGET_SR:.1f}s @ {TARGET_SR} Hz | rolloff {roll:.0f} Hz | HF {hf:.1f}% | SNR {snr:.1f} dB")
        print(f"     transcript: \"{text[:88]}{'...' if len(text) > 88 else ''}\"")
    except Exception as e:
        skipped.append((vid, str(e)))
        shutil.rmtree(vdir, ignore_errors=True)
        print(f"❌ {vid}: {e}")
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

print(f"\n🎉 Installed {len(installed)} premium voices: {installed}")
if skipped:
    print(f"⚠️  Skipped {len(skipped)}: {[s[0] for s in skipped]}")
# Release Whisper before the API server cell runs. Even on CPU this frees a
# few hundred MB of RAM; if anyone switches it back to CUDA this is what keeps
# the GPU clear for Fish Speech.
try:
    del _whisper
except NameError:
    pass
import gc
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"🧹 GPU free after cleanup: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
except Exception:
    pass

print("\n👉 Now re-run the API server cell below so it re-scans references/.")


In [ ]:
# 3b. (Optional) Install an EMOTIONAL VARIANT pack
#
# Fish has no emotion parameter and no tag parser -- expression is carried by
# the reference clip itself. So an "emotion" is a sibling reference of the same
# speaker, named Speaker__style:
#
#     Aria__neutral   Aria__happy   Aria__sad   Aria__whisper
#
# AetherStudio groups those in the voice picker and lets a script switch
# between them mid-run with [happy] / [sad] markers, which the app resolves
# into reference ids before sending. See docs/FISH_VOICE_STUDIO_SPEC.md.
#
# Source: Expresso (ylacombe/expresso) -- professional voice actors recorded in
# a studio at 48 kHz, with the emotional style labelled per utterance, which is
# exactly the shape this needs. Its clips are single sentences, so several
# same-speaker/same-style utterances are joined to reach a usable reference
# length. Transcripts come from the dataset itself, so they are ground truth --
# no Whisper pass and no chance of the .lab drifting from the .wav.
# --- Environment + storage detection (Colab vs Kaggle vs local) ---
# Kaggle has no /content, and caps /kaggle/working at 20 GB. The S2 Pro model
# does not fit there via git-lfs (which stores every file twice), so the model
# goes to whichever mount actually has free space and is symlinked in.
import os, shutil

if os.path.isdir('/kaggle'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
FISH_DIR = os.path.join(BASE, 'fish-speech')
os.makedirs(BASE, exist_ok=True)

# Pick the roomiest scratch mount for the ~10 GB of model weights.
_cands = []
for _c in ('/kaggle/temp', '/tmp', os.path.join(BASE, '.scratch')):
    try:
        os.makedirs(_c, exist_ok=True)
        _cands.append((shutil.disk_usage(_c).free, _c))
    except Exception:
        pass
SCRATCH = max(_cands)[1] if _cands else BASE
os.environ['HF_HOME'] = os.path.join(SCRATCH, 'hf')

def disk_report(label=''):
    print(f'💾 Disk {label}')
    for _p in dict.fromkeys([BASE, SCRATCH, '/']):
        try:
            _t, _u, _f = shutil.disk_usage(_p)
            print(f'   {_p:<24} {_f/1e9:6.1f} GB free / {_t/1e9:6.1f} GB total')
        except Exception:
            pass

print(f'📁 Environment base: {BASE}')
print(f'📁 Fish Speech dir : {FISH_DIR}')
print(f'📁 Model scratch   : {SCRATCH}')

import os, io, shutil, math
import requests

if not os.path.isdir(FISH_DIR):
    raise RuntimeError(f'{FISH_DIR} does not exist - run the install cell (cell 1) first.')

# ---- choose your speaker and styles ----------------------------------------
SPEAKER      = "ex01"        # ex01..ex04 in Expresso
VOICE_NAME   = "Aria"        # what it will be called in AetherStudio
STYLES       = ["default", "happy", "sad", "whisper", "confused"]
TARGET_SEC   = 12.0          # aim per reference
MAX_ROWS     = 400           # how far to scan the dataset
# -----------------------------------------------------------------------------

!pip install -q soundfile librosa 2>/dev/null
import numpy as np, soundfile as sf, librosa

TARGET_SR = 44100
DS = "ylacombe/expresso"
BASE_URL = "https://datasets-server.huggingface.co/rows"

print(f"🔎 Scanning {DS} for speaker {SPEAKER}...")
buckets = {}   # style -> list of (audio_url, text)
offset = 0
while offset < MAX_ROWS:
    r = requests.get(BASE_URL, params={
        "dataset": DS, "config": "read", "split": "train",
        "offset": offset, "length": 100
    }, timeout=60)
    if not r.ok:
        print(f"⚠️  dataset server returned {r.status_code}; stopping scan.")
        break
    rows = r.json().get("rows", [])
    if not rows:
        break
    for item in rows:
        row = item.get("row", {})
        if row.get("speaker_id") != SPEAKER:
            continue
        style = (row.get("style") or "default").strip().lower()
        if style not in STYLES:
            continue
        audio = row.get("audio")
        url = None
        if isinstance(audio, list) and audio:
            url = audio[0].get("src")
        elif isinstance(audio, dict):
            url = audio.get("src")
        if not url:
            continue
        buckets.setdefault(style, []).append((url, (row.get("text") or "").strip()))
    offset += 100

print("   found:", {k: len(v) for k, v in buckets.items()})

ref_root = os.path.join(FISH_DIR, "references")
os.makedirs(ref_root, exist_ok=True)
installed = []

for style, items in buckets.items():
    if not items:
        continue
    clips, texts, total = [], [], 0.0
    for url, text in items:
        if total >= TARGET_SEC:
            break
        try:
            raw = requests.get(url, timeout=60).content
            y, _ = librosa.load(io.BytesIO(raw), sr=TARGET_SR, mono=True)
        except Exception as e:
            print(f"   skip clip ({e})")
            continue
        if len(y) / TARGET_SR < 0.4:      # too short to contribute
            continue
        clips.append(y)
        # a short gap keeps the joined utterances from running together
        clips.append(np.zeros(int(0.25 * TARGET_SR), dtype=np.float32))
        if text:
            texts.append(text.rstrip(" .") + ".")
        total += len(y) / TARGET_SR + 0.25

    if total < 4.0:
        print(f"❌ {style}: only {total:.1f}s available - skipped.")
        continue

    # A reference with no transcript is unusable: Fish drives in-context
    # learning from the .lab, and its loader rejects an empty one, so the voice
    # installs cleanly and then fails every request with a 500 about a second
    # in. Rows in this dataset can carry an empty text field, which previously
    # produced exactly that -- audio long enough to pass the check above, joined
    # to "". Refuse the style instead of installing a voice that cannot work.
    joined = " ".join(texts).strip()
    if not joined:
        print(f"❌ {style}: audio is fine but the dataset rows carried no")
        print(f"   transcript, so the .lab would be empty - skipped.")
        continue

    y = np.concatenate(clips)
    peak = float(np.max(np.abs(y))) or 1.0
    y = y * (10 ** (-1.0 / 20.0) / peak)       # peak-normalise to -1 dBFS

    vid = f"{VOICE_NAME}__{style}"
    vdir = os.path.join(ref_root, vid)
    shutil.rmtree(vdir, ignore_errors=True)
    os.makedirs(vdir, exist_ok=True)
    sf.write(os.path.join(vdir, "audio.wav"), y, TARGET_SR, subtype="PCM_16")
    with open(os.path.join(vdir, "audio.lab"), "w", encoding="utf-8") as f:
        f.write(joined)

    installed.append(vid)
    print(f"✅ {vid}: {total:.1f}s from {len(texts)} utterance(s)")
    print(f"     \"{joined[:90]}...\"")

if installed:
    print(f"\n🎭 Installed {len(installed)} emotional variants: {installed}")
    print(f"   In AetherStudio pick \"{VOICE_NAME}\" and write e.g. [happy] or [sad]")
    print("   mid-script to switch delivery per passage.")
else:
    print("\n⚠️  Nothing installed - the dataset server may be unavailable, or the")
    print("   speaker/style names may have changed. The premium pack is unaffected.")

print("\n👉 Re-run the API server cell below so it re-scans references/.")


In [ ]:
# 4. Launch Unified API Server
import nest_asyncio
import uvicorn
import torch
import httpx
import subprocess
import time
import requests
import psutil
from fastapi import FastAPI, Request
from pydantic import BaseModel   # request schema for /v1/align
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.background import BackgroundTask
from pyngrok import ngrok

# --- Environment + storage detection (Colab vs Kaggle vs local) ---
# Kaggle has no /content, and caps /kaggle/working at 20 GB. The S2 Pro model
# does not fit there via git-lfs (which stores every file twice), so the model
# goes to whichever mount actually has free space and is symlinked in.
import os, shutil

if os.path.isdir('/kaggle'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
FISH_DIR = os.path.join(BASE, 'fish-speech')
os.makedirs(BASE, exist_ok=True)

# Pick the roomiest scratch mount for the ~10 GB of model weights.
_cands = []
for _c in ('/kaggle/temp', '/tmp', os.path.join(BASE, '.scratch')):
    try:
        os.makedirs(_c, exist_ok=True)
        _cands.append((shutil.disk_usage(_c).free, _c))
    except Exception:
        pass
SCRATCH = max(_cands)[1] if _cands else BASE
os.environ['HF_HOME'] = os.path.join(SCRATCH, 'hf')

def disk_report(label=''):
    print(f'💾 Disk {label}')
    for _p in dict.fromkeys([BASE, SCRATCH, '/']):
        try:
            _t, _u, _f = shutil.disk_usage(_p)
            print(f'   {_p:<24} {_f/1e9:6.1f} GB free / {_t/1e9:6.1f} GB total')
        except Exception:
            pass

print(f'📁 Environment base: {BASE}')
print(f'📁 Fish Speech dir : {FISH_DIR}')
print(f'📁 Model scratch   : {SCRATCH}')

if not os.path.isdir(FISH_DIR):
    raise RuntimeError(
        f'{FISH_DIR} does not exist — run the install cell (cell 1) first.'
    )
os.chdir(FISH_DIR)

nest_asyncio.apply()

# Ensure any zombie ngrok tunnels from previous interrupted runs are killed
ngrok.kill()
os.system("killall -9 ngrok 2>/dev/null")
os.system("pkill -9 -f \"tools.api_server\" || true")
for conn in psutil.net_connections():
    if conn.laddr.port in [8000, 8081] and conn.status == 'LISTEN':
        try:
            psutil.Process(conn.pid).terminate()
        except:
            pass
time.sleep(1)

# Show which voices exist BEFORE boot — the server only scans ./references once,
# relative to its own cwd, so a mismatch here is why the app shows no voices.
ref_root = os.path.join(FISH_DIR, "references")
found = sorted(
    d for d in os.listdir(ref_root)
    if os.path.isdir(os.path.join(ref_root, d))
) if os.path.isdir(ref_root) else []
print(f"🎙️  Reference voices detected in {ref_root}: {found if found else 'NONE'}")
if not found:
    print("⚠️  No reference voices — run the voice-pack cell above, then re-run THIS cell.")

# --- GPU preflight ------------------------------------------------------
# Fish needs ~13.8 GB of the T4's 14.56 GB: the LLAMA weights are ~12.8 GB and
# the DAC decoder allocates 1.0 GB on top. There is well under a gigabyte of
# slack, so anything this kernel is still holding will crash the decoder load.
try:
    import torch, gc
    gc.collect()
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No GPU is attached to this session. Fish Speech S2 Pro needs one — "
            "on CPU it takes minutes per line and the model load alone runs ~4 "
            "minutes. Enable it under Settings ▸ Accelerator (Kaggle) or "
            "Runtime ▸ Change runtime type (Colab), then re-run from cell 1. "
            "If the accelerator is already on, your weekly GPU quota is likely "
            "spent."
        )
    if True:
        torch.cuda.empty_cache()
        _free, _total = torch.cuda.mem_get_info()
        print(f"🎮 GPU: {_free/1e9:.2f} GB free of {_total/1e9:.2f} GB")
        if _free / 1e9 < 13.5:
            print("⚠️  Under 13.5 GB free — the decoder needs 1.0 GB after a ~12.8 GB")
            print("   model load. If boot fails with CUDA OOM, restart the kernel")
            print("   (Run ▸ Restart session) and run cells 1, 3, 4 without re-running")
            print("   anything that puts a model on the GPU.")
except Exception as _e:
    print(f"(GPU preflight skipped: {_e})")

print("🐟 Starting Fish Speech S2 Pro API Server (Subprocess)...")
# expandable_segments reduces allocator fragmentation, which matters when the
# decoder's 1.0 GB request has to fit in the sliver left after the model load.
_env = dict(os.environ)
_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
fish_process = subprocess.Popen(
    ["python", "-m", "tools.api_server", "--listen", "127.0.0.1:8081", "--half"],
    cwd=FISH_DIR,  # CRITICAL: references/ is resolved relative to this directory
    env=_env,
)

print("⏳ Waiting for Fish Speech to boot (Takes ~2 mins)...")
while True:
    if fish_process.poll() is not None:
        raise RuntimeError(
            f"Fish Speech server exited early with code {fish_process.returncode}. "
            "Scroll up for its traceback."
        )
    try:
        if requests.get("http://127.0.0.1:8081/v1/health", timeout=5).status_code == 200:
            print("✅ Fish Speech Ready!")
            break
    except:
        pass
    time.sleep(5)

try:
    _refs = requests.get(
        "http://127.0.0.1:8081/v1/references/list?format=json", timeout=10
    ).json()
    print(f"✅ API reports these voices: {_refs.get('reference_ids')}")
except Exception as e:
    print(f"⚠️  Could not read voice list from the API: {e}")


app = FastAPI(title="AETHER All-in-One")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

client = httpx.AsyncClient(base_url="http://127.0.0.1:8081", timeout=None)

# Headers that describe a single hop's framing/encoding. They MUST NOT be copied
# between the upstream response and our own — Starlette sets its own framing for a
# StreamingResponse, so passing the upstream Content-Length / Transfer-Encoding
# through makes browsers truncate or reject the audio stream.
HOP_BY_HOP = {
    "connection", "keep-alive", "proxy-authenticate", "proxy-authorization",
    "te", "trailers", "transfer-encoding", "upgrade",
    "content-length", "content-encoding",
}


# ---- Forced alignment -------------------------------------------------------
#
# Fish Speech returns audio and nothing else — no word timings, no phonemes.
# But this notebook already loads faster_whisper for voice-clone
# transcription, and faster_whisper can emit word-level timestamps. So the
# alignment the renderer needs costs no new dependency: transcribe the audio
# Fish just produced and read the word boundaries back out.
#
# Without this, the app can only measure the length of a whole chunk and
# distribute words inside it by syllable weight — accurate at the chunk edges,
# approximate within. With it, every gesture, highlight, chart reveal and
# mouth shape can land on the exact word.
#
# Whisper stays on the CPU deliberately. It shares this box with Fish's model
# on a 16GB card, and putting it on the GPU is what caused the CUDA OOM at
# boot earlier. Alignment is not latency-critical; correctness is.
_aligner = None


def _get_aligner():
    """Load the alignment model once, lazily. Loading costs a few seconds."""
    global _aligner
    if _aligner is None:
        from faster_whisper import WhisperModel
        _aligner = WhisperModel("small.en", device="cpu", compute_type="int8")
        print("  [align] whisper small.en loaded on CPU")
    return _aligner


class AlignReq(BaseModel):
    audio_b64: str = ""      # wav/mp3 bytes, base64
    text: str = ""           # what was spoken, when known — improves accuracy


@app.post("/v1/align")
async def align(req: AlignReq):
    """Word-level timings for a piece of narration.

    Returns { duration, words: [{text, start, end}], source }. The app treats
    this as its master clock; anything it cannot align falls back to measured
    chunk durations rather than guessing.
    """
    import base64 as _b64
    import tempfile as _tf
    import os as _os

    if not req.audio_b64:
        return JSONResponse({"error": "audio_b64 is required"}, status_code=400)

    path = None
    try:
        raw = _b64.b64decode(req.audio_b64)
        fd, path = _tf.mkstemp(suffix=".wav")
        with _os.fdopen(fd, "wb") as fh:
            fh.write(raw)

        model = _get_aligner()
        segments, info = model.transcribe(
            path,
            language="en",
            beam_size=5,
            word_timestamps=True,          # the whole point of this endpoint
            # A known transcript keeps Whisper honest about wording it would
            # otherwise "correct", which would break phrase lookup in the app.
            initial_prompt=(req.text or None),
        )

        words = []
        for seg in segments:
            for w in (getattr(seg, "words", None) or []):
                token = (w.word or "").strip()
                if not token:
                    continue
                words.append({
                    "text": token,
                    "start": round(float(w.start), 3),
                    "end": round(float(w.end), 3),
                })

        duration = round(float(getattr(info, "duration", 0) or (words[-1]["end"] if words else 0)), 3)
        return {
            "source": "aligned" if words else "empty",
            "duration": duration,
            "words": words,
            "phonemes": [],   # faster_whisper stops at words; visemes derive from these
        }
    except Exception as e:
        import traceback
        traceback.print_exc()
        return JSONResponse({"error": f"{type(e).__name__}: {e}"}, status_code=500)
    finally:
        if path:
            try:
                _os.unlink(path)
            except Exception:
                pass


@app.get("/aether/status")
async def aether_status():
    """Quick diagnostics you can open directly in a browser."""
    try:
        r = await client.get("/v1/references/list?format=json")
        voices = r.json().get("reference_ids", [])
    except Exception as e:
        voices = f"error: {e}"
    return JSONResponse({
        "ok": True,
        "base": BASE,
        "fish_dir": FISH_DIR,
        "cwd": os.getcwd(),
        "references_dir": ref_root,
        "voices": voices,
        "alignment": True,          # POST /v1/align returns word timings
    })


@app.api_route("/v1/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS"])
async def proxy_fish(path: str, request: Request):
    url = httpx.URL(path=request.url.path, query=request.url.query.encode("utf-8"))
    # Forward the client's headers minus framing ones, and ask upstream for an
    # unencoded body so the bytes we stream out match the headers we send.
    fwd = {
        k: v for k, v in request.headers.items()
        if k.lower() not in HOP_BY_HOP and k.lower() not in ("host", "accept-encoding")
    }
    fwd["accept-encoding"] = "identity"

    req = client.build_request(
        request.method, url, headers=fwd, content=await request.body()
    )
    res = await client.send(req, stream=True)

    safe_headers = {
        k: v for k, v in res.headers.items() if k.lower() not in HOP_BY_HOP
    }
    return StreamingResponse(
        res.aiter_raw(),
        status_code=res.status_code,
        headers=safe_headers,
        media_type=res.headers.get("content-type"),
        # Without this the upstream response is never released and the connection
        # pool leaks until the tunnel stops answering.
        background=BackgroundTask(res.aclose),
    )

public_url = ngrok.connect(8000).public_url
print("\n" + "="*60)
print("🚀 AETHER ALL-IN-ONE API IS LIVE")
print("="*60)
print(f"  Paste this ONE link into the FISH SPEECH Settings box in AetherStudio:")
print(f"  URL: {public_url}")
print(f"  Sanity check in your browser: {public_url}/aether/status")
print("="*60)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
import asyncio
await server.serve()